<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week1/week1_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# EYE OF RA — Standard Environment Header
# Paste this as the FIRST cell in every new notebook
# Run it before anything else
# ============================================================

# Install libraries not pre-loaded in Colab
# (runs fast after first time — Colab caches these)
!pip install -q pdfplumber shap xgboost streamlit plotly \
    openpyxl folium streamlit-folium camelot-py

# ============================================================
# Core imports — used across the entire project
# ============================================================

# Data handling
import pandas as pd
import numpy as np

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

# Explainability
import shap

# PDF extraction
import pdfplumber

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# System
import os
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================================
print("=" * 50)
print("✅ Eye of Ra environment ready")
print("All systems go. Let's catch some corruption. 🇬🇭")
print("=" * 50)

✅ Eye of Ra environment ready
All systems go. Let's catch some corruption. 🇬🇭


In [9]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Step 1: Verify all our PDF files are here and readable
# ============================================================

import os
import pdfplumber

# List all PDF files we uploaded
pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print(f"✅ Found {len(pdf_files)} PDF files\n")
print("=" * 50)
for f in pdf_files:
    print(f"  📄 {f}")
print("=" * 50)

✅ Found 14 PDF files

  📄 ppa_annual_report_2024.pdf
  📄 ppa_bulletin_2021_nov_dec.pdf
  📄 ppa_bulletin_2022_jul_aug.pdf
  📄 ppa_bulletin_2022_mar_apr.pdf
  📄 ppa_bulletin_2022_nov_dec.pdf
  📄 ppa_bulletin_2023_jul_aug.pdf
  📄 ppa_bulletin_2023_mar_apr.pdf
  📄 ppa_bulletin_2023_may_jun.pdf
  📄 ppa_bulletin_2023_nov_dec.pdf
  📄 ppa_bulletin_2023_sep_oct.pdf
  📄 ppa_bulletin_2024_jan_feb.pdf
  📄 ppa_bulletin_2024_mar_apr.pdf
  📄 ppa_bulletin_2024_may_jun.pdf
  📄 ppa_bulletin_2024_nov_dec.pdf


In [10]:
# ============================================================
# DIAGNOSTIC — find the broken file
# ============================================================

import os

pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print("Checking file sizes...\n")
print("=" * 50)

for filename in pdf_files:
    filepath = f'/content/{filename}'
    size_kb = os.path.getsize(filepath) / 1024
    print(f"{filename}")
    print(f"   Size: {size_kb:.1f} KB")
    print()

print("=" * 50)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add diagnostic cell - check file sizes"
# ------------------------------------------------------------

Checking file sizes...

ppa_annual_report_2024.pdf
   Size: 18147.6 KB

ppa_bulletin_2021_nov_dec.pdf
   Size: 8721.1 KB

ppa_bulletin_2022_jul_aug.pdf
   Size: 2512.9 KB

ppa_bulletin_2022_mar_apr.pdf
   Size: 2463.5 KB

ppa_bulletin_2022_nov_dec.pdf
   Size: 6175.7 KB

ppa_bulletin_2023_jul_aug.pdf
   Size: 5243.7 KB

ppa_bulletin_2023_mar_apr.pdf
   Size: 2630.5 KB

ppa_bulletin_2023_may_jun.pdf
   Size: 7390.5 KB

ppa_bulletin_2023_nov_dec.pdf
   Size: 11851.6 KB

ppa_bulletin_2023_sep_oct.pdf
   Size: 1934.2 KB

ppa_bulletin_2024_jan_feb.pdf
   Size: 1688.9 KB

ppa_bulletin_2024_mar_apr.pdf
   Size: 5600.8 KB

ppa_bulletin_2024_may_jun.pdf
   Size: 4395.8 KB

ppa_bulletin_2024_nov_dec.pdf
   Size: 8258.1 KB



In [11]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 2: Verify all PDF files are uploaded and readable
# (with error handling for corrupted uploads)
# ============================================================

pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print(f"Found {len(pdf_files)} PDF files. Checking each one...\n")
print("=" * 50)

good_files = []
bad_files = []

for filename in pdf_files:
    filepath = f'/content/{filename}'
    try:
        with pdfplumber.open(filepath) as pdf:
            num_pages = len(pdf.pages)
            first_page_text = pdf.pages[0].extract_text()

            if first_page_text:
                preview = first_page_text[:100].replace('\n', ' ')
                print(f"✅ {filename}")
                print(f"   Pages: {num_pages}")
                print(f"   Preview: {preview}...")
                good_files.append(filename)
            else:
                print(f"⚠️  {filename}")
                print(f"   Pages: {num_pages}")
                print(f"   WARNING: No readable text on page 1")
                bad_files.append(filename)
    except Exception as e:
        print(f"❌ {filename}")
        print(f"   ERROR: {str(e)[:80]}")
        bad_files.append(filename)
    print()

print("=" * 50)
print(f"✅ Readable files: {len(good_files)}")
print(f"❌ Problem files:  {len(bad_files)}")

if bad_files:
    print("\nThese files need to be re-uploaded:")
    for f in bad_files:
        print(f"  → {f}")

print("=" * 50)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 2 - PDF verification with error handling"
# ------------------------------------------------------------

Found 14 PDF files. Checking each one...

✅ ppa_annual_report_2024.pdf
   Pages: 86
   Preview: EXECUTIVE SUMMARY Introduction This Annual Report has been prepared in fulfillment of sections 3 (i)...

✅ ppa_bulletin_2021_nov_dec.pdf
   Pages: 16
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2021 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2022_jul_aug.pdf
   Pages: 12
   Preview: Public Procurement Authority: Electronic Bulletin Jul-Aug 2022 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2022_mar_apr.pdf
   Pages: 11
   Preview: Public Procurement Authority: Electronic Bulletin Mar-Apr 2022 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2022_nov_dec.pdf
   Pages: 14
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2022 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2023_jul_aug.pdf
   Pages: 23
   Preview: Public Procurement Authority: Electronic Bulletin Jul-Aug 2023 Submit 2023 Procurement Plan Us